# Poison-Resilient Continual Intrusion Detection on CICDDoS2019: results storyboard

**Question.** A detector that keeps learning new attacks replays stored examples of old ones. If an attacker flips
the labels stored in that replay buffer, can a **validation gate** plus a **frozen-teacher repair** keep the detector
safe *and* still learning?

**What this notebook does.**
- It reads only result files **committed to the public repository**. It does not train, does not score the
  held-out test, and needs no GPU, raw CSVs, checkpoints or zips.
- It redraws the seven report figures and checks every plotted number against the committed result tables.
- Every figure is labelled **VALIDATION**, **HELD-OUT TEST** or **DATA PROVENANCE**. Post-hoc material is marked
  **post hoc**.

**Scope.**
- The defended continual result covers **four labels** (BENIGN, Syn, LDAP, UDP) and uses FU2 teacher relabelling,
  a **post-hoc** method.
- The 12-label continual goal is **unsolved**.
- RF is a three-label, fixed-class limitation study.

**Run on Kaggle:** Settings → Internet **On**, Accelerator **None (CPU)**, no datasets → *Run All*.
**Run locally** (repository root): `python make_figures.py --execute-notebook` runs this notebook;
`python make_figures.py --check` confirms the plotted numbers match `docs/figures/*_data.csv` and the result tables.

In [ ]:
# 1. Setup: find the repository (local clone) or clone it from GitHub (Kaggle / anywhere else); safe to rerun
import importlib, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/DevChi1l/cicddos2019-continual-nids.git"
REPO_REF = os.environ.get("NIDS_REF", "main")  # branch or tag to read; committed files only


def _run(cmd, cwd=None):
    """Run quietly; show the captured output only if the command fails."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print("$", " ".join(cmd), "\n", r.stdout, r.stderr, file=sys.stderr)
        raise RuntimeError(f"command failed ({r.returncode}): {' '.join(cmd)}")
    return r.stdout.strip()


def _local_root():
    for d in (Path.cwd().resolve(), Path.cwd().resolve().parent):
        if (d / "make_figures.py").exists() and (d / "experiments" / "results").is_dir():
            return d
    return None


ROOT = _local_root()
if ROOT is None:  # not inside a clone: use (or create) one under /kaggle/working, else under the current folder
    base = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
    ROOT = base / "cicddos2019-continual-nids"
    if (ROOT / ".git").is_dir():
        try:  # rerun: refresh the existing clone to the requested ref
            _run(["git", "fetch", "--quiet", "--depth", "1", "origin", REPO_REF], cwd=ROOT)
            _run(["git", "reset", "--quiet", "--hard", "FETCH_HEAD"], cwd=ROOT)
        except RuntimeError:
            print("Could not refresh the clone (Internet off?); using the existing copy.")
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)  # an incomplete earlier clone of this repository folder
        _run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)])
    if not (ROOT / "make_figures.py").exists():
        raise SystemExit(f"{ROOT} does not contain make_figures.py; check REPO_URL / NIDS_REF.")

# figure code needs only pandas, numpy and matplotlib (preinstalled on Kaggle); install quietly if missing
missing = [m for m in ("pandas", "numpy", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    _run([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import pandas as pd
from IPython.display import Image, display, Markdown

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import make_figures as M
if Path(M.__file__).resolve().parent != ROOT:  # a different copy was imported earlier in this kernel
    sys.modules.pop("make_figures"); M = importlib.import_module("make_figures")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 80)

try:
    COMMIT = _run(["git", "rev-parse", "HEAD"], cwd=ROOT)
except (RuntimeError, FileNotFoundError):
    COMMIT = "unknown (not a git checkout)"
print("repository:", ROOT)
print("commit:    ", COMMIT)
print("figures are written to:", M.FIG)

## Committed inputs
Every input file must be present in the clone. The first 12 hex digits of each sha256 are shown for the record.

In [ ]:
# 2. Artifact / provenance check: every committed input exists; sha256 (first 12 hex) shown for the record
try:
    prov = M.provenance()
except M.MissingArtifact as e:
    print("Cannot build the storyboard:\n", e)
    raise
print(f"{len(prov)} committed input files present")
display(prov)

## Dataset and preprocessing

**Dataset.** CICDDoS2019 (Kaggle copy): 18 CSV files, 70,427,637 flow rows, 14 raw labels.

**Audit findings.**
- 59.9 million rows are within-file duplicates.
- File names do not match their contents, so the label column decides.
- BENIGN is only 0.16% of the rows.

**Preprocessing.**
1. Rows were deduplicated globally on the feature vector, and 235,261 label-conflicting vectors were dropped.
2. Every attack label was capped (BENIGN is never capped). The data was split 60/10/10/20 at random.
3. Each flow becomes 77 numeric features plus protocol. A scaler (signed log, standardise, clip ±5) is fitted on E0
   training rows only and then frozen.
4. Evaluation rows identical to a training row after scaling are excluded (166 rows in the four-label setting).


## Data pipeline (provenance)

**Claim:**
- All 70,427,637 raw rows (18 files, 14 labels) were read once.
- Global deduplication and label-conflict removal shrink every class by orders of magnitude.
- BENIGN is 0.16% of the rows and is never capped.

**Limitation:**
- Training used capped subsets: 98,833 rows in the four-label setting and at most 159,595 (the 12-label oracle).
- There is no full-data training claim.

In [ ]:
data_fig1_data = M.fig1_data()  # writes docs/figures/fig1_data_pipeline.png and fig1_data_pipeline_data.csv
display(Image(filename=str(M.FIG / "fig1_data_pipeline.png")))
display(data_fig1_data.head(40))

## Model

An **FT-Transformer-style** tabular Transformer (`s2_model.py`):
- Each of the 77 numeric features becomes a learned 32-dimensional token. A protocol token and a [CLS] token are
  added, giving 79 tokens.
- It has 3 PreNorm encoder layers with 4 attention heads, GELU and dropout 0.1. A linear head on [CLS] gives class
  logits.
- It has **43,396 parameters** with four classes. The head grows when a new class arrives, and the old output rows
  are copied unchanged.

It is not the original TabTransformer, which embeds only categorical columns. Details: `docs/ARCHITECTURE.md`.

## Continual protocol, threat, gate and repair

**Experiences (class-incremental, fresh BENIGN each time):**
- E0 = BENIGN + Syn
- E1 = + LDAP
- E2 = + UDP

A replay buffer keeps 2,000 rows per class from each accepted experience.

**Threat.** The attacker flips a fraction *f* of the **stored Syn labels** to BENIGN, in the buffer only. New data,
validation, test, weights and the teacher are untouched.

**One update at experience *t*:**
1. **C1:** train the last accepted model on new data + buffer.
2. **Gate (validation only):** old-class recall ≥ reference − 0.05, new-class recall ≥ 0.80, benign FAR ≤ 0.05,
   new-class FAR ≤ 0.05.
3. If C1 fails, **R1** retrains from the last accepted model, using the **frozen E0 teacher T** on the buffer:
   - *soft* (pre-registered): row weight = p_T(stored label);
   - *relabel* (FU2, **post hoc**): confidently wrong stored labels are replaced by T's prediction.
4. If R1 also fails, **keep the old model** (fallback). Only accepted data enters the buffer.

The held-out test is scored once, after the validation decisions are committed.

## Pre-registered four-class sweep

**Claim** (pre-registered; seeds 42-44):
- The gate rejects more poisoned updates as f grows [validation].
- Soft repair lets the model learn LDAP and UDP for f <= 0.95 [test].

**Limitation:**
- At f = 1.0 soft repair fails (0/6).
- The gate alone keeps a stale model that never learns LDAP or UDP.
- Hollow dots show that gate-only results are bimodal across seeds.

In [ ]:
data_fig2_sweep = M.fig2_sweep()  # writes docs/figures/fig2_sweep.png and fig2_sweep_data.csv
display(Image(filename=str(M.FIG / "fig2_sweep.png")))
display(data_fig2_sweep.head(40))

## FU2: four-class defence at 100% flips (PRIMARY, post hoc)

**Claim** [test]: at f = 1.0, teacher relabelling learns all four classes in 3/3 seeds, within about 0.01 of clean, with a benign FAR of about 0.0011.

**Limitation:**
- The method is post hoc: chosen after the sweep's diagnostic, and E1 replays that diagnostic.
- Four labels, one attacked class, 3 seeds.
- A FAR of 0 for gate-only is not safety: the stale model calls LDAP and UDP benign.

In [ ]:
data_fig3_fu2 = M.fig3_fu2()  # writes docs/figures/fig3_fu2_defence.png and fig3_fu2_defence_data.csv
display(Image(filename=str(M.FIG / "fig3_fu2_defence.png")))
display(data_fig3_fu2.head(40))

## 12-label continual vs joint oracle (LIMITATION)

**Claim** [test]:
- The CII continual 12-label models retain 6 of 12 classes.
- A joint oracle reaches all 12, but 8 or fewer reach recall 0.80.

**Limitation:**
- The oracle is one jointly trained diagnostic model, not a continual or defended result.
- The 12-label continual goal is unsolved.
- Continual runs are single-seed.

In [ ]:
data_fig4_oracle = M.fig4_oracle()  # writes docs/figures/fig4_12label_vs_oracle.png and fig4_12label_vs_oracle_data.csv
display(Image(filename=str(M.FIG / "fig4_12label_vs_oracle.png")))
display(data_fig4_oracle.head(40))

## RC / RCS clean pilots (validation only, stopped)

**Claim** [validation]: every clean rejection is an old non-Syn class forgotten when an overlapping class arrives.

**Limitation:**
- The registered stopping rules ended both experiments before any poisoned run.
- There is no test result to plot.
- Only three schedules were tried.

In [ ]:
data_fig5_pilots = M.fig5_pilots()  # writes docs/figures/fig5_rc_rcs_pilots.png and fig5_rc_rcs_pilots_data.csv
display(Image(filename=str(M.FIG / "fig5_rc_rcs_pilots.png")))
display(data_fig5_pilots.head(40))

## RF: confidence loss without recall loss (LIMITATION, post hoc)

**Claim** (validation and test in separate panels):
- At 10% more flips per update (60% after six updates), Syn recall holds at about 0.994.
- Mean Syn confidence falls to about 0.77.
- No update was rejected, so repair never triggered.

**Limitation:**
- The confidence loss was not repaired.
- Three fixed classes; traffic refresh, not new-class arrival.

In [ ]:
data_fig6_rf = M.fig6_rf()  # writes docs/figures/fig6_rf_trajectory.png and fig6_rf_trajectory_data.csv
display(Image(filename=str(M.FIG / "fig6_rf_trajectory.png")))
display(data_fig6_rf.head(40))

## Experiment status

Which experiments were pre-registered, post hoc, validated, tested once or stopped. Each cell is asserted against the evidence folders present in git.

In [ ]:
data_fig7_status = M.fig7_status()  # writes docs/figures/fig7_status.png and fig7_status_data.csv
display(Image(filename=str(M.FIG / "fig7_status.png")))
display(data_fig7_status.head(40))

## Plotted numbers vs committed result tables

In [ ]:
# 3. Every plotted number is checked against an independently read committed table (assertions raise on mismatch)
data = {f.__name__: globals()[f"data_{f.__name__}"] for f in M.FIGURES}
for m in M.verify(data):
    print("OK", m)
print("\nfigure files:", sorted(p.name for p in M.FIG.glob("fig*.png")))

## Summary

| Result | Split | Status |
|---|---|---|
| FU2 relabelling at 100% Syn flips learns all four classes (3/3 seeds) | held-out test | **post hoc** method; four labels only |
| Gate only / soft repair at 100% flips: LDAP = UDP = 0 | held-out test | registered controls |
| Soft repair learns all four classes for f ≤ 0.95 (9/9 vs 3/9 gate-only) | held-out test | pre-registered sweep |
| 12-label continual learning keeps 6 of 12 classes | held-out test | **limitation**: unsolved |
| RF: recall holds (~0.994) but Syn confidence falls to ~0.77; repair never ran | validation + test | **post hoc**, **limitation** |

FU2, the sweep and RF use seeds 42-44; the 12-label continual runs are single-seed. No statistical-significance
claim is made. The numbers above come from the figures and checks in this
notebook.